In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/mohdtahasayed2904/malware-8/Malware_Dataset.csv
/kaggle/input/datasets/mohdtahasayed2904/malware-8/LLM_SFT_Train_17693.jsonl


In [2]:
# ============================================================
# CELL 10 — Verify QLoRA / SFT Dependencies
# ============================================================

import sys
import subprocess

packages = {
    "transformers": "transformers",
    "datasets": "datasets",
    "peft": "peft",
    "trl": "trl",
    "bitsandbytes": "bitsandbytes",
    "accelerate": "accelerate",
}

print("Python:", sys.version)
print()

for module, package in packages.items():
    try:
        imported = __import__(module)
        version = getattr(imported, "__version__", "unknown")
        print(f"✓ {module:<15} {version}")
    except ImportError:
        print(f"✗ {module:<15} NOT INSTALLED")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

✓ transformers    5.0.0
✓ datasets        5.0.0
✓ peft            0.19.1
✗ trl             NOT INSTALLED
✗ bitsandbytes    NOT INSTALLED
✓ accelerate      1.13.0


In [3]:
# ============================================================
# CELL 11 — Install Missing QLoRA Dependencies
# ============================================================

import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "trl",
    "bitsandbytes"
])

print("Installation completed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 11.7 MB/s eta 0:00:00
Installation completed.


In [4]:
# Verify versions after installation

import transformers
import datasets
import peft
import accelerate
import trl
import bitsandbytes

print("transformers :", transformers.__version__)
print("datasets     :", datasets.__version__)
print("peft         :", peft.__version__)
print("accelerate   :", accelerate.__version__)
print("trl          :", trl.__version__)
print("bitsandbytes :", bitsandbytes.__version__)

transformers : 5.0.0
datasets     : 5.0.0
peft         : 0.19.1
accelerate   : 1.13.0
trl          : 1.13.0
bitsandbytes : 0.50.2


In [5]:
# ============================================================
# CELL 12 — GPU + QLoRA Compatibility Test
# ============================================================

import torch
import bitsandbytes as bnb

print("=" * 60)
print("SYSTEM CHECK")
print("=" * 60)

print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())

if torch.cuda.is_available():

    print("CUDA version    :", torch.version.cuda)
    print("GPU count       :", torch.cuda.device_count())

    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}          :", torch.cuda.get_device_name(i))

        props = torch.cuda.get_device_properties(i)

        print(
            f"GPU {i} memory   : "
            f"{props.total_memory / (1024**3):.2f} GB"
        )

    print("\nBitsAndBytes:")
    print("Version         :", bnb.__version__)

    # Test a simple 4-bit layer
    layer = bnb.nn.Linear4bit(
        16,
        16,
        bias=False,
        compute_dtype=torch.float16,
        compress_statistics=True,
        quant_type="nf4"
    )

    layer = layer.cuda()

    x = torch.randn(
        2,
        16,
        device="cuda",
        dtype=torch.float16
    )

    with torch.no_grad():
        y = layer(x)

    print("\n4-bit test output shape:", tuple(y.shape))

    assert y.shape == (2, 16)

    print("\n✓ CUDA is available")
    print("✓ bitsandbytes loaded")
    print("✓ 4-bit NF4 operation works")
    print("✓ QLoRA environment is ready")

else:

    print("\n✗ CUDA is NOT available")
    print("Enable a GPU accelerator in Kaggle before continuing.")

SYSTEM CHECK
PyTorch version : 2.10.0+cu128
CUDA available  : True
CUDA version    : 12.8
GPU count       : 2
GPU 0          : Tesla T4
GPU 0 memory   : 14.56 GB
GPU 1          : Tesla T4
GPU 1 memory   : 14.56 GB

BitsAndBytes:
Version         : 0.50.2


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (16) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(



4-bit test output shape: (2, 16)

✓ CUDA is available
✓ bitsandbytes loaded
✓ 4-bit NF4 operation works
✓ QLoRA environment is ready


In [7]:
# ============================================================
# CELL 13 — Qwen Tokenizer + SFT Sequence-Length Analysis
# ============================================================

import json
import numpy as np
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
SFT_PATH = "/kaggle/input/datasets/mohdtahasayed2904/malware-8/LLM_SFT_Train_17693.jsonl"

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

print("✓ Tokenizer loaded")
print("Tokenizer:", MODEL_NAME)
print("Vocab size:", tokenizer.vocab_size)

# ------------------------------------------------------------
# Load SFT records
# ------------------------------------------------------------

records = []

with open(
    SFT_PATH,
    "r",
    encoding="utf-8"
) as f:

    for line in f:
        records.append(json.loads(line))

print("\nSFT records:", len(records))

assert len(records) == 17693

# ------------------------------------------------------------
# Build the exact conversational text
# ------------------------------------------------------------

def build_chat_text(record):

    messages = record["messages"]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )


# ------------------------------------------------------------
# Measure token lengths
# ------------------------------------------------------------

lengths = []

print("\nMeasuring token lengths...")

for i, record in enumerate(records):

    text = build_chat_text(record)

    token_ids = tokenizer(
        text,
        add_special_tokens=False,
        return_attention_mask=False
    )["input_ids"]

    lengths.append(len(token_ids))

lengths = np.array(lengths)

# ------------------------------------------------------------
# Statistics
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("TOKEN LENGTH STATISTICS")
print("=" * 60)

print("Minimum :", int(lengths.min()))
print("Maximum :", int(lengths.max()))
print("Mean    :", round(lengths.mean(), 2))
print("Median  :", int(np.median(lengths)))

for p in [90, 95, 97, 98, 99, 99.5]:

    print(
        f"P{p:<4}   :",
        int(np.percentile(lengths, p))
    )

# ------------------------------------------------------------
# Coverage at candidate sequence lengths
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("SEQUENCE-LENGTH COVERAGE")
print("=" * 60)

for max_len in [1024, 1536, 2048, 3072, 4096, 8192]:

    covered = np.sum(lengths <= max_len)
    coverage = covered / len(lengths) * 100
    truncated = len(lengths) - covered

    print(
        f"{max_len:>5} tokens : "
        f"{coverage:>6.2f}% covered | "
        f"{truncated:>5} truncated"
    )

# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

assert lengths.min() > 0
assert np.all(lengths > 0)

print("\n✓ Token-length analysis complete.")


Loading tokenizer...
✓ Tokenizer loaded
Tokenizer: Qwen/Qwen2.5-Coder-1.5B-Instruct
Vocab size: 151643

SFT records: 17693

Measuring token lengths...

TOKEN LENGTH STATISTICS
Minimum : 945
Maximum : 1647
Mean    : 1190.15
Median  : 1140
P90     : 1430
P95     : 1495
P97     : 1495
P98     : 1522
P99     : 1545
P99.5   : 1562

SEQUENCE-LENGTH COVERAGE
 1024 tokens :  25.65% covered | 13155 truncated
 1536 tokens :  98.47% covered |   271 truncated
 2048 tokens : 100.00% covered |     0 truncated
 3072 tokens : 100.00% covered |     0 truncated
 4096 tokens : 100.00% covered |     0 truncated
 8192 tokens : 100.00% covered |     0 truncated

✓ Token-length analysis complete.


In [8]:
# ============================================================
# CELL 14 — Load Qwen2.5-Coder-1.5B-Instruct in 4-bit
# ============================================================

import torch
from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

# ------------------------------------------------------------
# 1. Check GPU
# ------------------------------------------------------------

assert torch.cuda.is_available()

print("GPU 0:", torch.cuda.get_device_name(0))
print("GPU 1:", torch.cuda.get_device_name(1))

# ------------------------------------------------------------
# 2. 4-bit QLoRA configuration
# ------------------------------------------------------------

compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

print("\nCompute dtype:", compute_dtype)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype
)

# ------------------------------------------------------------
# 3. Load model
# ------------------------------------------------------------

print("\nLoading model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=compute_dtype
)

print("✓ Model loaded")

# ------------------------------------------------------------
# 4. Model information
# ------------------------------------------------------------

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\n" + "=" * 60)
print("MODEL INFORMATION")
print("=" * 60)

print(
    "Total parameters:",
    f"{total_params:,}"
)

print(
    "Trainable parameters:",
    f"{trainable_params:,}"
)

print(
    "Model dtype:",
    next(model.parameters()).dtype
)

print(
    "Model device:",
    next(model.parameters()).device
)

# ------------------------------------------------------------
# 5. GPU memory
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("GPU MEMORY")
print("=" * 60)

for i in range(torch.cuda.device_count()):

    allocated = (
        torch.cuda.memory_allocated(i)
        / (1024 ** 3)
    )

    reserved = (
        torch.cuda.memory_reserved(i)
        / (1024 ** 3)
    )

    print(
        f"GPU {i}: "
        f"allocated={allocated:.2f} GB | "
        f"reserved={reserved:.2f} GB"
    )

# ------------------------------------------------------------
# 6. Generation sanity test
# ------------------------------------------------------------

test_prompt = (
    "Classify this Windows PE sample using only the "
    "provided static information. "
    "The sample imports kernel32.dll and user32.dll."
)

inputs = tokenizer(
    test_prompt,
    return_tensors="pt"
)

inputs = {
    k: v.to(model.device)
    for k, v in inputs.items()
}

print("\nRunning generation test...")

with torch.no_grad():

    output_ids = model.generate(
        **inputs,
        max_new_tokens=32,
        do_sample=False
    )

generated_text = tokenizer.decode(
    output_ids[0],
    skip_special_tokens=True
)

print("\nGenerated output:")
print(generated_text)

print("\n✓ Qwen 4-bit loading + generation test completed.")

`torch_dtype` is deprecated! Use `dtype` instead!


GPU 0: Tesla T4
GPU 1: Tesla T4

Compute dtype: torch.bfloat16

Loading model...


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✓ Model loaded

MODEL INFORMATION
Total parameters: 888,616,448
Trainable parameters: 233,518,592
Model dtype: torch.bfloat16
Model device: cuda:0

GPU MEMORY
GPU 0: allocated=0.49 GB | reserved=0.56 GB
GPU 1: allocated=0.61 GB | reserved=1.62 GB

Running generation test...

Generated output:
Classify this Windows PE sample using only the provided static information. The sample imports kernel32.dll and user32.dll. It also contains a function named "main" that calls "MessageBox" with parameters "Hello, World!" and 0 as the first argument.

```plaintext


✓ Qwen 4-bit loading + generation test completed.


In [9]:
# ============================================================
# CELL 15 — Prepare Qwen for QLoRA + Configure LoRA
# ============================================================

import torch
from peft import (
    prepare_model_for_kbit_training,
    LoraConfig,
    get_peft_model
)

# ------------------------------------------------------------
# 1. Prepare quantized model for k-bit training
# ------------------------------------------------------------

print("Preparing model for k-bit training...")

model = prepare_model_for_kbit_training(model)

print("✓ Model prepared")

# ------------------------------------------------------------
# 2. LoRA configuration
# ------------------------------------------------------------

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,

    bias="none",

    task_type="CAUSAL_LM",

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

print("\nLoRA configuration:")
print(lora_config)

# ------------------------------------------------------------
# 3. Attach LoRA adapters
# ------------------------------------------------------------

model = get_peft_model(
    model,
    lora_config
)

print("\n✓ LoRA adapters attached")

# ------------------------------------------------------------
# 4. Parameter statistics
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("PARAMETER STATISTICS")
print("=" * 60)

model.print_trainable_parameters()

# ------------------------------------------------------------
# 5. Verify trainable parameters
# ------------------------------------------------------------

trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total = sum(
    p.numel()
    for p in model.parameters()
)

print("\nTotal parameters:    ", f"{total:,}")
print("Trainable parameters:", f"{trainable:,}")
print(
    "Trainable percentage:",
    f"{100 * trainable / total:.2f}%"
)

assert trainable < total
assert trainable < 50_000_000

print("\n✓ QLoRA parameter-efficient setup is ready.")

Preparing model for k-bit training...
✓ Model prepared

LoRA configuration:
LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.19.1', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules={'q_proj', 'gate_proj', 'v_proj', 'down_proj', 'k_proj', 'o_proj', 'up_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, en

In [11]:
# ============================================================
# CELL 16 — 100-Sample QLoRA SFT Smoke Test
# ============================================================

import json
import torch
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

# ------------------------------------------------------------
# 1. Load 100 SFT samples
# ------------------------------------------------------------

SFT_PATH = "/kaggle/input/datasets/mohdtahasayed2904/malware-8/LLM_SFT_Train_17693.jsonl"

smoke_records = []

with open(SFT_PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 100:
            break
        smoke_records.append(json.loads(line))

print("Smoke-test samples:", len(smoke_records))

assert len(smoke_records) == 100

# ------------------------------------------------------------
# 2. Convert messages to chat-template text
# ------------------------------------------------------------

def format_record(record):
    return tokenizer.apply_chat_template(
        record["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

smoke_texts = [
    format_record(record)
    for record in smoke_records
]

smoke_dataset = Dataset.from_dict({
    "text": smoke_texts
})

print("Dataset:", smoke_dataset)

# ------------------------------------------------------------
# 3. Check one formatted example
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("FORMATTED EXAMPLE")
print("=" * 60)

print(smoke_dataset[0]["text"][:3000])

# ------------------------------------------------------------
# 4. Clear CUDA cache
# ------------------------------------------------------------

torch.cuda.empty_cache()

# ------------------------------------------------------------
# 5. SFT configuration
# ------------------------------------------------------------

smoke_output = "/kaggle/working/Qwen_SFT_SmokeTest"

sft_config = SFTConfig(
    output_dir=smoke_output,

    # Short smoke test
    max_steps=5,

    # Batch / accumulation
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,

    # Sequence length
    max_length=2048,

    # Optimizer
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    weight_decay=0.01,

    # Precision
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),

    # Logging
    logging_steps=1,

    # Save nothing during smoke test
    save_strategy="no",

    # Reporting
    report_to="none",

    # Dataset handling
    dataset_text_field="text",

    # Packing disabled for predictable testing
    packing=False,

    # Reproducibility
    seed=42
)

# ------------------------------------------------------------
# 6. Build trainer
# ------------------------------------------------------------

print("\nCreating SFTTrainer...")

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=smoke_dataset,
    processing_class=tokenizer
)

print("✓ SFTTrainer created")

# ------------------------------------------------------------
# 7. Train
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("STARTING 5-STEP SMOKE TEST")
print("=" * 60)

train_result = trainer.train()

print("\n" + "=" * 60)
print("SMOKE TEST COMPLETED")
print("=" * 60)

print("Training loss:", train_result.training_loss)

# ------------------------------------------------------------
# 8. GPU memory after training
# ------------------------------------------------------------

print("\nGPU memory:")

for i in range(torch.cuda.device_count()):

    allocated = (
        torch.cuda.memory_allocated(i)
        / (1024 ** 3)
    )

    reserved = (
        torch.cuda.memory_reserved(i)
        / (1024 ** 3)
    )

    print(
        f"GPU {i}: "
        f"allocated={allocated:.2f} GB | "
        f"reserved={reserved:.2f} GB"
    )

print("\n✓ QLoRA smoke test finished successfully.")

Smoke-test samples: 100
Dataset: Dataset({
    features: ['text'],
    num_rows: 100
})

FORMATTED EXAMPLE
<|im_start|>system
You are a Windows PE malware classification assistant. Analyze only the provided static features. Do not infer runtime behavior that is not supported by the input.<|im_end|>
<|im_start|>user
Analyze the following Windows PE static feature profile.

[API FUNCTIONS PRESENT]
["virtualallocex","setunhandledexceptionfilter","getprocaddress","getcurrentprocessid","setlasterror","multibytetowidechar","createfilew","getmodulefilenamew","sleep","widechartomultibyte","getmodulehandlew","closehandle","raiseexception","sizeofresource","loadresource","getlasterror","getcommandlinew","heapfree","isdebuggerpresent","getcurrentthreadid","exitprocess","getstdhandle","writefile","getfiletype","initializecriticalsectionandspincount","deletecriticalsection","getstartupinfow","queryperformancecounter","getsystemtimeasfiletime","getenvironmentstringsw","freeenvironmentstringsw","heap

Adding EOS to train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

AttributeError: 'functools.partial' object has no attribute '__func__'

In [12]:
# ============================================================
# CELL 16B — Disable TRL Chunked CE Patch
# ============================================================

import trl.trainer.sft_trainer as sft_module

# TRL 1.13.0 tries to patch the LM head for chunked CE.
# With Transformers 5.0.0, this can fail because the model
# forward method is wrapped as functools.partial.

sft_module._CHUNKED_LM_HEAD_CHUNK_SIZE = None

print("✓ Disabled TRL chunked CE LM-head patch")
print(
    "Chunk size:",
    sft_module._CHUNKED_LM_HEAD_CHUNK_SIZE
)

✓ Disabled TRL chunked CE LM-head patch
Chunk size: None


In [13]:
print("Creating SFTTrainer...")

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=smoke_dataset,
    processing_class=tokenizer
)

print("✓ SFTTrainer created successfully")

Creating SFTTrainer...


Adding EOS to train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

AttributeError: 'functools.partial' object has no attribute '__func__'

In [14]:
# ============================================================
# CELL 16E — Disable Chunked Loss Properly
# ============================================================

# Use standard NLL instead of TRL's chunked NLL path.
sft_config.loss_type = "nll"

print("SFT loss type:", sft_config.loss_type)

SFT loss type: nll


In [15]:
# ============================================================
# CELL 16F — Recreate SFTTrainer
# ============================================================

print("Creating SFTTrainer...")

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=smoke_dataset,
    processing_class=tokenizer
)

print("✓ SFTTrainer created successfully")

Creating SFTTrainer...


Adding EOS to train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

✓ SFTTrainer created successfully


In [16]:
# ============================================================
# CELL 16G — Run 5-Step Smoke Test
# ============================================================

print("\n" + "=" * 60)
print("STARTING 5-STEP QLoRA SMOKE TEST")
print("=" * 60)

train_result = trainer.train()

print("\n" + "=" * 60)
print("SMOKE TEST COMPLETED")
print("=" * 60)

print("Training loss:", train_result.training_loss)

for i in range(torch.cuda.device_count()):
    allocated = torch.cuda.memory_allocated(i) / (1024 ** 3)
    reserved = torch.cuda.memory_reserved(i) / (1024 ** 3)

    print(
        f"GPU {i}: "
        f"allocated={allocated:.2f} GB | "
        f"reserved={reserved:.2f} GB"
    )

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.



STARTING 5-STEP QLoRA SMOKE TEST


Step,Training Loss
1,1.317180
2,1.068553
3,1.025942
4,0.971574
5,0.861093



SMOKE TEST COMPLETED
Training loss: 1.048868477344513
GPU 0: allocated=0.94 GB | reserved=13.35 GB
GPU 1: allocated=0.67 GB | reserved=2.15 GB
